# 🚨 This AI Can't Write a Single Word, and That's Why It's So Fast

We build a **scam message detector** with **Jev**, a new kind of AI model from TypeSafe AI that doesn't write text at all. It only **decides**.

Then we race it against a regular AI chatbot on 100 messages, find its one dangerous flaw, and fix it.

**How to read this notebook**
- 🧑‍🏫 **Teach**: the cells that matter. This is where Jev replaces a regular LLM.
- ▶️ **Just run**: plumbing we prepared for you (timing, scoring, the web app). Run it and look at the results.

> ⚠️ A teaching project, not a safety product. Never treat a model's answer as proof that a message is safe.

## Step 0 ▶️ Just run: setup

You need **one** key from [OpenRouter](https://openrouter.ai/settings/keys), with a few dollars of credit (this notebook uses well under $1).
Add it in Colab's 🔑 **Secrets** panel as `OPENROUTER_API_KEY` and switch on notebook access.

In [ ]:
!pip install -q typesafe-sdk openai gradio pandas matplotlib
!wget -q -nc https://raw.githubusercontent.com/Kiran-B23/jev-scam-detector/main/jev_helpers.py
!mkdir -p data && wget -q -nc -P data https://raw.githubusercontent.com/Kiran-B23/jev-scam-detector/main/data/scam_messages.csv

from jev_helpers import *
KEY = get_api_key()

## Step 1 🧑‍🏫 Teach: the old way, asking a chatbot

Here's how most people would build a scam check today: ask an LLM, ask for JSON, then read the answer.
Watch **exactly** what comes back.

In [ ]:
sms = ("Dear consumer, your electricity power will be disconnected tonight at 9:30 PM "
       "because previous month bill was not updated. Call officer 9XXXX X7788 immediately.")

reply = ask_llm('Is this SMS a scam? Reply in JSON like {"is_scam": true}.\n\n' + sms)
print(repr(reply))

In [ ]:
import json
try:
    print("Parsed:", json.loads(reply))   # 🤞 hope it's clean JSON, with the right key, and a real true/false
except json.JSONDecodeError as e:
    print("💥 json.loads crashed:", e)

Whatever you got, notice that **the chatbot gave us text**, and our code has to *hope* that text is in the right shape.
Run Step 1 a few times: the format can change from run to run.

## Step 2 🧑‍🏫 Teach: the swap, asking Jev instead

Jev doesn't write. You give it:
- a **state**: the thing to judge (our SMS)
- **questions**: typed questions, each with a name you choose

It gives back **numbers**, always in the shape you asked for.

In [ ]:
from typesafe_sdk import TypeSafeClient, Noul, Choice, Score

jev = TypeSafeClient(api_key=KEY, base_url="https://openrouter.ai/api", model="jev-1.13")

answer = jev.system_one(
    state=sms,
    questions={"is_scam": Noul(instructions="Is this SMS a scam?")},
)
answer.nouls["is_scam"].noul

That number is the **probability the answer is yes**. `0.97` means "very likely a scam". `0.5` means "I'm not sure".
There's no JSON to parse, nothing to hope about.

## Step 3 🧑‍🏫 Teach: three questions, one call

Jev has three question types, like the bubbles on an OMR sheet:

| Type | Asks | Returns |
|---|---|---|
| **Noul** | yes or no? | probability of yes |
| **Choice** | which one? | the best option + a probability for every option |
| **Score** | how much? | a position on your scale |

We ask one of each, in a **single** call.

In [ ]:
SCAM_TYPES = {
    "fake_refund_or_upi": "Fake refund, cashback or 'scan QR / enter PIN to receive money'",
    "kyc_or_account_block": "KYC, PAN, SIM or bank account 'will be blocked'",
    "job_or_task": "Part-time job or task that needs a fee or deposit",
    "prize_or_lottery": "Lottery, lucky draw, prize or reward points",
    "police_or_parcel_threat": "Courier, customs, police or 'digital arrest' threats",
    "bill_disconnection": "Electricity, gas, broadband or mobile 'will be cut' today",
    "fake_family_or_boss": "Pretends to be family, a friend or a boss and asks for money or codes",
    "investment_or_loan": "Guaranteed returns, stock tips, crypto or instant loans",
    "not_scam": "A normal, genuine message",
}

QUESTIONS = {
    "is_scam": Noul(
        instructions="Is this message a scam or fraud attempt?",
        criteria={"true": "Tries to trick the reader into paying, sharing a code, clicking or calling",
                  "false": "A genuine message, even if it mentions OTPs, money or deadlines"},
    ),
    "scam_type": Choice(instructions="What kind of message is this?", criteria=SCAM_TYPES),
    "pressure": Score(instructions="How much pressure does it put on the reader?",
                      criteria=["None", "Mild deadline", "Strong urgency", "Threats"]),
}

answer = jev.system_one(state=sms, questions=QUESTIONS)
print("Scam?    ", answer.nouls["is_scam"].noul)
print("Type:    ", answer.choices["scam_type"].choice)
print("Pressure:", answer.scores["pressure"].score, "out of 3")

## Step 4 🧑‍🏫 Teach: turn the probability into a decision

A probability isn't an action. **We** decide how sure Jev must be before we act, like DRS in cricket: clear cases get decided, and close ones are "umpire's call".

In [ ]:
def check(message):
    a = jev.system_one(state=message, questions=QUESTIONS)
    p = a.nouls["is_scam"].noul
    if p >= 0.90:
        verdict = "🚨 Likely scam"
    elif p <= 0.10:
        verdict = "✅ Looks normal"
    else:
        verdict = "⚠️ Not sure. Double-check"
    return {"verdict": verdict, "p_scam": p, "scam_type": a.choices["scam_type"].choice,
            "type_probs": dict(a.choices["scam_type"].probabilities), "pressure": a.scores["pressure"].score}

for msg in [sms,
            "123456 is your OTP for login. Do not share it with anyone. Bank staff will never ask for your OTP.",
            "Hi dear, I sent a code to your number by mistake, can you please forward it to me? It's urgent."]:
    r = check(msg)
    print(f"{r['verdict']:<28} p={r['p_scam']:.2f}  {r['scam_type']:<24} | {msg[:60]}...")

## Step 5 ▶️ Just run: the race, Jev vs a regular LLM on 100 messages

100 made-up messages (60 scams, 40 genuine, about a third deliberately tricky). Both models get the **same** questions and the **same** rules: 10 messages at a time.

In [ ]:
df = load_messages()
results = await race(df, QUESTIONS, SCAM_TYPES)

In [ ]:
scoreboard(results)

## Step 6 ▶️ Just run: find the flaw

Jev **can't** give a broken answer. But can it give a **wrong** one, and be sure about it?

In [ ]:
confident_mistakes(results)

In [ ]:
threshold_table(results)      # stricter threshold = fewer mistakes, more "second checks"
# reliability_plot(results)   # optional: when Jev says 80%, are ~80% really scams?

## Step 7 🧑‍🏫 Teach: the fix, Jev decides and the chatbot explains

Jev handles **every** message, fast and cheap. Only when it's **not sure** do we ask the slower chatbot for a second opinion in plain words, which is the one thing Jev can't do.

In [ ]:
def check_with_backup(message):
    result = check(message)
    if result["verdict"].startswith("⚠️"):
        result["explanation"] = ask_llm(
            "In 2 short sentences of simple English, say whether this SMS looks like a scam and what to do next. "
            "Never tell the reader to click links or call numbers in it.\n\nSMS: " + message)
    return result

unsure = results[(results.p_scam_jev < 0.90) & (results.p_scam_jev > 0.10)]
print(len(unsure), "of 100 messages would get a second opinion.")
if len(unsure):
    check_with_backup(unsure.message.iloc[0])

## Step 8 ▶️ Just run: ship it as a web app

In [ ]:
launch_app(check_with_backup, examples=[
    [sms],
    ["Hi Mom, this is my new number, my phone fell in water. Can you send Rs 8,000 urgently?"],
    ["Your order has been shipped and will arrive by Thursday. Track it in the app."],
])

## What we learned
- **Chatbots write. Jev decides.** For yes/no, pick-one and rate-it questions, swapping in Jev removes all the parsing, and it's faster and cheaper.
- **Valid ≠ correct.** Jev's answers are always in shape, but they can still be wrong, and sure of it.
- **Thresholds + a backup** make it trustworthy: act only when confident, and ask for a second opinion otherwise.
- **Test the claims yourself.** We did, on 100 messages.

**Your turn:** add five tricky messages of your own to `data/scam_messages.csv` (remove personal details first) and rerun Steps 5 and 6.